# register_embryos — cohort walkthrough

One cohort, ND2 files to registered atlas.

A **cohort** is every embryo sharing `(genotype, timepoint, view, magnification)`.
That 4-tuple names the output directory. Gene panel is deliberately *not* part of the
key: embryos of the same genotype and stage imaged the same way belong in one
registered space even when the partner genes differ.

The two steps that used to be hand-edited literals — rotation angles and contrast
limits — happen in one widget (section 3) and are saved as JSON, so every later run
reproduces them without re-deciding anything.

Slow steps are flagged. Each `wf.<step>()` stores its result on `wf`, so a later step
can be re-run with different parameters without repeating the earlier ones.

## 0. Setup

In [ ]:
%matplotlib inline
from datetime import datetime
from pathlib import Path

import register_embryos as re
from register_embryos import CohortWorkflow

ND2_DIR  = Path("/path/to/nd2")                    # <- edit
OUT_ROOT = Path("/path/to/output") / datetime.now().strftime("%Y%m%d")
COHORT   = "wt_12s_dorsal_20X"                     # <- from `re.scan()` below

print("register_embryos", re.__version__)
print("open3d ICP backend available:", re.HAS_OPEN3D)
print("filename spec:", re.FILENAME_SPEC)
print("outputs ->", OUT_ROOT / COHORT)

## 1. What cohorts are here?

Run this first — it is the cheapest way to catch a misnamed file, which otherwise
shows up as a one-embryo cohort that cannot be registered.

If a filename is not on the spec yet:

```bash
register-embryos rename DIR --timepoint 12s            # dry run
register-embryos rename DIR --timepoint 12s --apply    # writes an undo manifest
```

In [ ]:
cohorts = re.scan(ND2_DIR)

## 2. Load — the slow first step

Reads each ND2, max-projects `bin_size` z-planes into each bin, normalises every
channel to [0,1], and reads the voxel size from the file metadata.

Max rather than mean projection: HCR puncta are sparse and bright, and averaging one
over several mostly-empty planes dilutes it below the signal threshold.

**`bin_size` depends on how you will segment** — 7 for 2D, and **1 for 3D**, which
takes the whole stack unbinned. Do the widget at 7 either way: the rotation and
contrast you pick carry over, and it is far quicker to look at.

The raw `(Z,C,Y,X)` stack is released after binning — at ~1.5 GB per embryo a whole
cohort of them will not fit alongside Cellpose.

In [ ]:
wf = CohortWorkflow.from_directory(ND2_DIR, output_root=OUT_ROOT, cohort=COHORT)

wf.load(bin_size=7)          # SLOW

v = wf.volumes[0]
print(f"\n{v.embryo_id}")
print(f"  binned shape (Z_bins, Y, X): {v.shape}")
print(f"  voxel: {v.voxel.xy_um:.3f} x {v.voxel.xy_um:.3f} x {v.voxel.z_um:.3f} um")
print(f"  raw anisotropy    : {v.voxel.anisotropy:.2f}")
print(f"  binned anisotropy : {v.binned_voxel.anisotropy:.2f}")
print(f"  gene map: {v.gene_map}")

## 3. Prepare — rotation and contrast, one widget

Left: the rotated frame. Middle: its histogram with the contrast window shaded.
Right: **what segmentation will actually see**, with saturated/floored percentages.

- **Orientation** applies to the whole embryo (all channels, all z-planes). Use the
  ±90° nudges to get anterior pointing the same way across the cohort, then the
  slider to fine-tune. A warning appears if rotating inside the fixed canvas would
  clip signal — tick *grow canvas* if so.
- **Contrast** applies to the selected channel only. `Accept contrast` advances to
  the next channel or embryo, so a cohort is walked with one button.

Both are written to `orientation.json` and `contrast_limits.json` in the cohort
directory as you accept them. Re-opening this widget resumes from them.

In [ ]:
config = wf.prepare(transform="none")   # transform="log2" lifts dim puncta
config

### Resume: reload an already-computed configuration

Run this **instead** of the cell above if you have set these before. `wf.prepare()`
reloads automatically too — this is for getting the object without opening the
widget.

In [ ]:
from register_embryos.widgets import PrepConfig

config = PrepConfig.load(OUT_ROOT / COHORT)      # missing files -> empty, no error

print(f"orientations: {len(config.orientations)}   "
      f"contrast: {len(config.contrast)} embryo(s), "
      f"transform={config.contrast.transform!r}")

missing = config.contrast.missing(wf.volumes)
print(f"unset embryo-channels: {len(missing)}")
for eid, ch in missing[:8]:
    print(f"  {eid} ch{ch}")

### No ipywidgets? Same job, non-interactively

`auto_contrast_limits` defaults to the **90th** percentile, not the 1st, and the
reason matters. An HCR channel is mostly background — on a real 20× stack the median
normalised intensity is 0.004 and the 99.5th percentile only 0.098 — so `p1` sits at
the *bottom* of the background rather than above it, floors nothing, and the narrow
high limit then stretches the dim background shoulder to full scale. With `p1/p99.5`,
**36–41 % of pixels landed above the 0.05 signal threshold** and essentially every
nucleus read as expressing. Check the `[NN%+]` fraction it prints.

In [ ]:
# from register_embryos import OrientationSet, auto_contrast_limits
#
# # the old `angles = [215, 155, 165, 70]`, but recorded:
# orientations = OrientationSet.from_angles(
#     [v.embryo_id for v in wf.volumes], [215, 155, 165, 70]
# )
# contrast = auto_contrast_limits(wf.volumes)      # prints a positive fraction per channel

### QC — rotation and contrast together

Pass **`config`**, not `config.contrast`: `config` carries the rotation too, so the
figure shows the rotated *and* contrasted frame. A preview that omitted the rotation
would let a wrong rotation pass unnoticed.

- **Rotation** — is anterior in the same direction everywhere? The title says what
  was applied.
- **% above 0.05** — the fraction segmentation will treat as signal. A few percent up
  to ~10% is plausible for a gene channel; 30%+ means background is being counted.
- **% saturated** — well under 1% for gene channels.

In [ ]:
from register_embryos import preview_contrast

for v in wf.volumes[:3]:
    preview_contrast(v, config,
                     save_path=OUT_ROOT / COHORT / "qc" / f"{v.embryo_id}_prepared.png")

## 4. Bake in the rotation and contrast

`auto_contrast=False` fails loudly if a channel is unset, rather than silently
filling it from percentiles.

In [ ]:
wf.apply_prep(config, auto_contrast=False)

print(f"\n{len(wf.adjusted)} volumes ready for segmentation")
print("history:", wf.adjusted[0].history)

## 5. Segment — the slow second step

| mode | z input | Cellpose runs | label ids | one nucleus becomes | signal assignment |
|---|---|---|---|---|---|
| `2d` | binned | once per z-plane | restart every plane | **one row per plane**, ids unrelated | within each plane |
| `2d+link` | binned | once per z-plane (*identical to `2d`*) | linked across planes by mask IoU | **one row**, true 3D centroid | within each plane |
| `3d` | **unbinned** | once over the volume (`do_3D` + anisotropy) | consistent by construction | **one row**, true 3D centroid | through the volume, in µm |

**`2d` and `2d+link` do exactly the same segmentation** — same Cellpose calls, same
masks, same cost. They differ only in *identity*: `2d+link` walks z and gives a label
the id of the label below it wherever their pixel IoU clears a threshold. That
matters because per-plane ids make one nucleus spanning two planes look like two
nuclei at nearly the same xy — duplicate points stacked in z, which ICP then fits as
if they were real structure. **If you are going to register, `2d+link` is strictly
better than `2d` at identical cost.**

What `2d+link` cannot do is split a blob that 2D already merged — only a genuine 3D
pass can — and it keeps per-plane assignment rather than 3D's µm-aware territory.

**3D takes the whole z-stack, unbinned, and this is enforced.** Binning is a
concession for 2D: it max-projects planes so each carries enough signal to segment
alone, which is exactly the information 3D works from. At `bin_size=7` (1.5 µm
z-step ⇒ 10.5 µm per plane) a ~6 µm nucleus spans 0.57 planes, so `do_3D` has nothing
to link. `segment(mode="3d")` raises on a binned volume.

Two separate knobs, often confused:

- **`anisotropy`** — z:xy voxel aspect ratio, from the ND2 × the binning factor.
  Cellpose rescales z with it so a spherical nucleus looks spherical. Too low ⇒
  nuclei split along z; too high ⇒ they merge.
- **`diameter`** — nucleus size in **xy pixels**. `None` lets Cellpose estimate it.

In [ ]:
wf.segment(mode="2d+link", diameter=None, gpu=False, max_workers=2)   # SLOW

for s in wf.segmented:
    print(f"  {s.embryo_id}: {s.n_labels} labels  (labels_are_3d={s.labels_are_3d})")

In [ ]:
# For 3D: reload unbinned first, then segment on a GPU.
# Unbinned is heavy (~0.8 GB per channel as float32 for a 200-plane 1024^2 stack,
# and Cellpose needs several multiples of that), so keep max_workers=1.
#
# wf.load(bin_size=1)
# wf.apply_prep(config)
# wf.segment(mode="3d", gpu=True, max_workers=1)

## 6. Assign signal pixels, build the nucleus table

HCR signal sits *around* nuclei, not inside the nuclear stain, so measuring only
inside the Cellpose mask throws most of it away. Each above-threshold pixel is given
to its nearest nucleus and the per-nucleus value is the mean over that territory.

Pixels dropped as background become a sentinel (`0.3`), not `0`, and the mean
excludes exactly that sentinel — a pixel that was never measured must not read as a
measured zero, or every nucleus mean is dragged toward zero in proportion to how much
empty space its territory covers.

In 3D mode distances are in **micrometres**, so an anisotropic stack does not
preferentially assign along z.

In [ ]:
wf.build_tables(signal_threshold=0.05)

print()
print(wf.combined.head())
print("\ncolumns:", list(wf.combined.columns))
print(f"unique nucleus ids: {wf.combined['nucleus_id'].nunique():,} "
      f"across {len(wf.combined):,} rows")

## 7. Register — point-to-point ICP onto a cohort reference

Each embryo is aligned to one reference, so excluding an embryo never changes how the
others land — only the atlas built from them.

Registration downsamples **uniformly in space** first: sampling uniformly at random
keeps dense regions dense, and ICP would then fit those and ignore the sparse ones.
Each axis is normalised to [0,1] first so z (a few dozen bins) is weighted like x and
y (a thousand pixels).

A PCA coarse alignment runs before ICP — ICP is a local method, and two embryos
mounted 90° apart will not find each other from a cold start.

Residuals use **the same metric on both sides**. Comparing a mean nearest-neighbour
distance against an RMS one makes a good fit look like a regression, because RMS of a
positive quantity always exceeds its mean.

In [ ]:
wf.register(
    reference_embryo_id=None,          # None = the first embryo in the cohort
    n_downsample=5000,
    max_correspondence_distance=500,   # xy pixels, calibrated to 1024x1024
    max_iteration=500,
)

print()
print(wf.registration.stats.to_string(index=False))

### Registration QC — always look

A residual cannot reveal an embryo that converged to a plausible-looking wrong pose.
Six panels per embryo: XY/XZ/YZ raw, then registered, reference in grey underneath.

Note XZ/YZ are **not** forced to equal aspect — x is in xy pixels and z in bin
indices, and forcing equal across incommensurate units flattens the embryo into a
pancake that reads as a property of the sample. Pass `z_aspect=<anisotropy>` for true
proportions.

In [ ]:
from register_embryos import plot_registration_2d

plot_registration_2d(
    wf.registration.registered, wf.registration.reference_embryo_id,
    mode="light", suptitle=f"{COHORT} — ICP QC",
);

## 8. Atlas — the composite embryo

At each anchor point (a reference-embryo nucleus), average the **k nearest nuclei
pooled across every embryo** — position and gene intensity both.

Choosing `k` is a real trade-off. With N embryos, `k ≈ N` averages roughly one nucleus
per embryo: it smooths between-embryo variability while preserving spatial detail.
Larger `k` blurs domain boundaries.

`atlas_diagnostics` reports the neighbour radius and how many distinct embryos
contributed — the only honest way to say whether an atlas point is a consensus or
just one embryo.

In [ ]:
wf.build_atlas(k_neighbors=None,      # None = the embryo count
               n_points=None)         # None = one point per reference nucleus

from register_embryos import atlas_diagnostics
print()
print(atlas_diagnostics(wf.atlas).T.to_string())

In [ ]:
# Leave-one-out: no atlas point may draw on its own embryo. Use this if you
# intend to ask how well the reference agrees with the atlas -- otherwise it is
# partly predicting itself.
#
# from register_embryos import build_atlas
# loo = build_atlas(wf.registration.registered,
#                   reference_embryo_id=wf.registration.reference_embryo_id,
#                   k_neighbors=4, exclude_self_embryo=True)

## 9. Plots — day and night

Two colouring schemes for two questions:

- **`plot_pointcloud_3d`** — one panel per gene, hue ramp by intensity.
  Quantitative: *where is this gene on?*
- **`plot_additive_3d` / `plot_additive_2d`** — all genes at once; hue from the
  additive mix at full brightness, size and opacity carrying intensity.
  Qualitative: *which combinations occur where?*

Separating hue from brightness keeps a three-colour overlay readable — a nucleus
expressing one dim gene still shows that gene's hue instead of fading out.

A nucleus counts as expressing when **one** channel clears threshold, not when the
channel sum does; otherwise it could be coloured while being positive for nothing.

Every function takes `mode="dark"` or `mode="light"`, and the greys, outlines and gain
differ — colour that reads bright on black reads washed out on white.

In [ ]:
wf.plot_all(modes=("dark", "light"))   # both themes -> <cohort>/figures/

In [ ]:
from register_embryos import plot_additive_3d, plot_pointcloud_3d

plot_additive_3d(wf.atlas.points, mode="dark", coords=("x", "y", "z"),
                 title=f"{COHORT} atlas")

## 10. Record what produced this

In [ ]:
wf.save_manifest()
print()
for key, value in wf.outputs().summary().items():
    print(f"  {key}: {value}")

## 11. Comparing cohorts — align two atlases

Puts two cohorts (e.g. mutant vs wild type) in one coordinate frame.
`center_first=True` because atlases from different cohorts can sit in quite different
coordinate ranges.

In [ ]:
# from register_embryos import align_atlases
#
# aligned = align_atlases({"wt": wt_atlas, "pbx": pbx_atlas},
#                         reference_label="wt", center_first=True)
# print(aligned.stats.to_string(index=False))

## 12. Re-doing the cheap steps

Registration and the atlas need only the nucleus table, so a different reference,
`k`, or embryo exclusion costs seconds — no images, no Cellpose:

```bash
register-embryos atlas out/<cohort>/combined_nucleus_table.csv \
    -o out/retry --k 6 --exclude <embryo_id>
```

## Running the whole thing as a job

The widget output is on disk, so this reproduces your accepted rotation and contrast
exactly:

```bash
register-embryos run <ND2_DIR> -o <OUT> --cohort <COHORT> \
    --mode 2d+link --workers 4 --no-auto-contrast
```

`--no-auto-contrast` makes it fail rather than quietly substituting percentiles.

For 3D on a GPU node (SGE), which reloads unbinned automatically:

```bash
qsub -q <queue> docs/qsub_gpu_segmentation.sh <ND2_DIR> <OUT> <COHORT>
```

## Project-specific extras

Steps that depend on the biology of a particular experiment live in
`register_embryos.contrib` and are **not** part of the standard workflow:

```python
from register_embryos.contrib import midline_filter
clean, bounds = midline_filter(wf.atlas.points, marker="wt1a", axis="y")
```